# 03_gnn_text_v2 — SOTA Traffic GNN

Upgrade over v1:
- **Gated TCN** — gated dilated convolutions for temporal
- **Diffusion GCN** — K-hop spatial aggregation
- **Adaptive adjacency** — learned road connectivity
- **Cross-modal fusion** — multi-head attention over text tokens
- **Qwen3.5-0.8B** — modern LLM text encoder (frozen + LoRA)

## Section 1: Setup & Data

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import time

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency,
    build_windows, compute_norm_stats, normalize, denormalize,
    train_val_split, compute_mse, write_submission, validate_submission,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj_raw = load_adjacency()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
X_norm = normalize(X_all, mean, std)
Y_norm = normalize(Y_all, mean, std)

print(f"X: {X_norm.shape}, Y: {Y_norm.shape}")
print(f"X range: [{X_norm.min():.2f}, {X_norm.max():.2f}]")

In [ ]:
X_tr, _, Y_tr, X_va, _, Y_va = train_val_split(X_norm, None, Y_norm, val_frac=0.2)
T_tr = T_all[:len(X_tr)]
T_va = T_all[len(X_tr):]
print(f"Train: {X_tr.shape}, Val: {X_va.shape}")

In [ ]:
def build_adj_tensor(adj_matrix):
    adj = torch.tensor(adj_matrix, dtype=torch.float32)
    deg = adj.sum(dim=1)
    deg_inv_sqrt = torch.pow(deg, -0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0
    return deg_inv_sqrt.diag() @ adj @ deg_inv_sqrt.diag()

adj_t = build_adj_tensor(adj_raw).to(DEVICE)
print(f"Adj: {adj_t.shape}, nnz={(adj_t > 0).sum().item()}")

### Text Encoders

Two options, both pre-computed and cached:
1. **MiniLM** (baseline) — `all-MiniLM-L6-v2`, 384-dim, ~22M params
2. **Qwen3.5** (SOTA) — `Qwen/Qwen3.5-0.8B`, 1024-dim, frozen + LoRA

In [ ]:
print("Encoding with MiniLM...")
from sentence_transformers import SentenceTransformer

minilm = SentenceTransformer("all-MiniLM-L6-v2")
T_emb_tr_minilm = minilm.encode(T_tr.tolist(), batch_size=256,
                                  show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
T_emb_va_minilm = minilm.encode(T_va.tolist(), batch_size=256,
                                  show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
print(f"MiniLM: train={T_emb_tr_minilm.shape}, val={T_emb_va_minilm.shape}")

In [ ]:
print("Encoding with Qwen3.5-0.8B (frozen + LoRA)...")
from transformers import AutoModel, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

QWEN_ID = "Qwen/Qwen3.5-0.8B"
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_ID, trust_remote_code=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_base = AutoModel.from_pretrained(QWEN_ID, trust_remote_code=True,
                                       torch_dtype=torch.float16, device_map=DEVICE)
qwen_base.eval()
for p in qwen_base.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
)
qwen_model = get_peft_model(qwen_base, lora_config)
qwen_model.print_trainable_parameters()

In [ ]:
@torch.no_grad()
def encode_qwen(texts, tokenizer, model, device, batch_size=16):
    model.eval()
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=256,
                           return_tensors="pt").to(device)
        outputs = model(**inputs, output_hidden_states=True)
        hidden = outputs.hidden_states[-1]  # (B, T, 1024)
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        embeddings.append(pooled.cpu().numpy())
    return np.concatenate(embeddings, axis=0).astype(np.float32)


T_emb_tr_qwen = encode_qwen(T_tr.tolist(), qwen_tokenizer, qwen_model, DEVICE)
T_emb_va_qwen = encode_qwen(T_va.tolist(), qwen_tokenizer, qwen_model, DEVICE)
print(f"Qwen3.5: train={T_emb_tr_qwen.shape}, val={T_emb_va_qwen.shape}")

In [ ]:
# Save embeddings to disk for reuse across notebook restarts
import pickle

cache = {
    "minilm_tr": T_emb_tr_minilm, "minilm_va": T_emb_va_minilm,
    "qwen_tr": T_emb_tr_qwen,   "qwen_va": T_emb_va_qwen,
    "mean": mean, "std": std,
}
with open("../dataset-task1/text_emb_cache.pkl", "wb") as f:
    pickle.dump(cache, f)
print("Embeddings cached to dataset-task1/text_emb_cache.pkl")

### DataLoader Helpers

In [ ]:
class TrafficDS(Dataset):
    def __init__(self, X, T_emb, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.T = torch.tensor(T_emb, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.T[i], self.Y[i]


def make_loaders(T_emb_tr, T_emb_va, batch_size=32):
    train_ds = TrafficDS(X_tr, T_emb_tr, Y_tr)
    val_ds = TrafficDS(X_va, T_emb_va, Y_va)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    return train_loader, val_loader


print("Section 1 complete.")

## Section 2: Model Components

All defined inline as `nn.Module` classes.

In [ ]:
class GatedTCN(nn.Module):
    """Stack of gated dilated 1D convolutions with skip connections."""
    def __init__(self, in_steps=15, residual_channels=64, skip_channels=64,
                 num_layers=4, kernel_size=2, dropout=0.1):
        super().__init__()
        self.num_layers = num_layers
        self.skip_convs = nn.ModuleList()
        self.filter_convs = nn.ModuleList()
        self.gate_convs = nn.ModuleList()

        in_dim = residual_channels
        for i in range(num_layers):
            dilation = 2 ** i
            padding = (kernel_size - 1) * dilation
            self.filter_convs.append(
                nn.Conv1d(in_dim, residual_channels, kernel_size,
                          dilation=dilation, padding=padding))
            self.gate_convs.append(
                nn.Conv1d(in_dim, residual_channels, kernel_size,
                          dilation=dilation, padding=padding))
            self.skip_convs.append(nn.Conv1d(residual_channels, skip_channels, 1))

        self.start_conv = nn.Conv1d(1, residual_channels, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.start_conv(x)
        skip_sum = 0

        for i in range(self.num_layers):
            residual = x
            f = torch.tanh(self.filter_convs[i](x))
            g = torch.sigmoid(self.gate_convs[i](x))
            x = f * g
            x = x[:, :, :residual.size(2)]
            x = x + residual[:, :, :x.size(2)]
            skip = self.skip_convs[i](x)
            skip_sum += skip
            x = self.dropout(x)

        return x[:, :, -1], skip_sum[:, :, -1:]  # (B*N, C), (B*N, skip, 1)


In [ ]:
class DiffusionGCN(nn.Module):
    """K-hop diffusion graph convolution."""
    def __init__(self, in_channels, out_channels, num_hops=2, dropout=0.1):
        super().__init__()
        self.num_hops = num_hops
        self.weights = nn.Parameter(
            torch.empty(num_hops, in_channels, out_channels))
        nn.init.xavier_uniform_(self.weights)
        self.norm = nn.LayerNorm(out_channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, adj):
        out = 0
        h = x
        for k in range(self.num_hops):
            h_k = h if k == 0 else adj @ h
            out_k = torch.einsum("bnc,co->bno", h_k, self.weights[k])
            out += out_k
            h = adj @ h
        return self.drop(F.relu(self.norm(out)))


In [ ]:
class AdaptiveAdj(nn.Module):
    """Learned adjacency matrix, combined with static road network."""
    def __init__(self, num_nodes=1260, embed_dim=10):
        super().__init__()
        self.src_emb = nn.Embedding(num_nodes, embed_dim)
        self.dst_emb = nn.Embedding(num_nodes, embed_dim)

    def forward(self, static_adj):
        N = static_adj.size(0)
        idx = torch.arange(N, device=static_adj.device)
        s = self.src_emb(idx)
        d = self.dst_emb(idx)
        a = F.softmax(F.relu(s @ d.T), dim=-1)
        deg = a.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0
        a_norm = deg_inv_sqrt.diag() @ a @ deg_inv_sqrt.diag()
        return static_adj + a_norm


In [ ]:
class CrossModalFusion(nn.Module):
    """Multi-head cross-attention: node features query text tokens."""
    def __init__(self, d_model, d_text, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_text, d_model)
        self.W_V = nn.Linear(d_text, d_model)
        self.W_O = nn.Linear(d_model, d_model)

        self.gate = nn.Linear(d_model * 2, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, node_feat, text_feat):
        B, N, _ = node_feat.shape
        S = text_feat.size(1)

        Q = self.W_Q(node_feat).reshape(B, N, self.n_heads, self.d_head)
        Q = Q.permute(0, 2, 1, 3)

        K = self.W_K(text_feat).reshape(B, S, self.n_heads, self.d_head)
        K = K.permute(0, 2, 1, 3)

        V = self.W_V(text_feat).reshape(B, S, self.n_heads, self.d_head)
        V = V.permute(0, 2, 1, 3)

        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)

        out = torch.matmul(attn, V)
        out = out.permute(0, 2, 1, 3).reshape(B, N, -1)
        out = self.W_O(out)

        combined = torch.cat([node_feat, out], dim=-1)
        g = torch.sigmoid(self.gate(combined))
        return g * node_feat + (1 - g) * out


In [ ]:
# Smoke test: instantiate and forward-pass each component
print("GatedTCN:")
tcn = GatedTCN(in_steps=15, residual_channels=64, skip_channels=64, num_layers=4, dropout=0.1)
x = torch.randn(10, 1, 15)
out_tcn, skip = tcn(x)
print(f"  Input: {x.shape} -> Output: {out_tcn.shape}, Skip sum: {skip.shape}")

print("DiffusionGCN:")
dgcn = DiffusionGCN(64, 64, num_hops=2, dropout=0.1)
adj_dummy = torch.eye(1260) * 0.5
x_g = torch.randn(2, 1260, 64)
out_g = dgcn(x_g, adj_dummy)
print(f"  Input: {x_g.shape} -> Output: {out_g.shape}")

print("AdaptiveAdj:")
aadj = AdaptiveAdj(1260, embed_dim=10)
adj_out = aadj(adj_dummy)
print(f"  Static: {adj_dummy.shape} -> Mixed: {adj_out.shape}")

print("CrossModalFusion:")
cmf = CrossModalFusion(d_model=64, d_text=384, n_heads=4, dropout=0.1)
node_f = torch.randn(2, 1260, 64)
text_f = torch.randn(2, 16, 384)
out_f = cmf(node_f, text_f)
print(f"  Node: {node_f.shape}, Text: {text_f.shape} -> Output: {out_f.shape}")

print("\nAll components OK.")
